# Simulation Results

This notebook explores the data logged by the simulation benchmark. It is separate from `paper_results.ipynb`, which analyzes human survey answers. Here the unit of analysis is a logged episode run from the simulator.

The goal is to make the simulation side easy to inspect before we connect it to the survey side: import the logs, check what was recorded, visualize the objective metrics, and run simple statistical comparisons between policies.

## What This File Uses

The local benchmark writes one or more `episode-metrics-*.csv` files under `artifacts/local-validation/`. Each CSV row represents one simulation episode run. The columns include technical status, normalized perceptual axes, and lower-level behavioral features.

The mapping used in the survey is kept here too:

| Survey block | Policy | POV | Meaning |
|---|---|---|---|
| Q1 | A | first person | The viewer is the objective/goal of the robot |
| Q2 | B | first person | The viewer is the objective/goal of the robot |
| Q3 | A | third person | The viewer observes the scene |
| Q4 | B | third person | The viewer observes the scene |

Each Q is composed of three episodes: from A to B with no obstacles, fixed obstacles, and moving people. In the simulation artifacts these are usually encoded as `ep1`, `ep2`, and `ep3` in the run directory name.

## Import

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ARTIFACT_ROOT = Path("artifacts/local-validation")
CSV_PATTERN = "episode-metrics-*.csv"

The benchmark metrics are split into two levels:

- **Axes** are the high-level quantities we want to compare with human perception.
- **Features** are the lower-level behavioral measurements used by the benchmark to build those axes.

All these columns are normalized scores where larger usually means better. Empty values mean that a metric did not apply to that episode, for example a human-distance feature in an episode without humans.

In [ ]:
AXIS_COLUMNS = [
    "perceived_dexterity",
    "perceived_safety",
    "perceived_social_awareness",
    "impression",
]

FEATURE_COLUMNS = [
    "task_success_rate",
    "task_completion_time",
    "comfort_aware_path_efficiency",
    "hesitation",
    "min_human_robot_distance",
    "proxemic_intrusion_dose",
    "speed_near_humans_p95",
    "gesture_response_success",
    "acknowledgement_clarity",
    "human_aware_approach",
    "bystander_ack",
    "sparc",
    "heading_jerk",
    "stability",
    "legibility",
    "behavioral_naturalness",
]

Q_MAPPING = {
    "q1": {"policy": "A", "pov": "first_person"},
    "q2": {"policy": "B", "pov": "first_person"},
    "q3": {"policy": "A", "pov": "third_person"},
    "q4": {"policy": "B", "pov": "third_person"},
}

EPISODE_LABELS = {
    "ep1": "no_obstacles",
    "ep2": "fixed_obstacles",
    "ep3": "moving_people",
}

SCORE_COLUMNS = AXIS_COLUMNS + FEATURE_COLUMNS

In [ ]:
def metadata_from_csv_path(path: Path) -> dict:
    """Derive survey/block metadata from the artifact directory layout."""
    relative_parts = path.relative_to(ARTIFACT_ROOT).parts
    survey_block = relative_parts[0] if relative_parts else None
    run_label = relative_parts[1] if len(relative_parts) > 1 else path.parent.name
    episode_match = re.search(r"ep(\d+)", run_label)
    episode_slot = f"ep{episode_match.group(1)}" if episode_match else None
    mapped = Q_MAPPING.get(survey_block, {})

    return {
        "survey_block": survey_block,
        "policy": mapped.get("policy"),
        "pov": mapped.get("pov"),
        "episode_slot": episode_slot,
        "episode_type": EPISODE_LABELS.get(episode_slot),
        "run_label": run_label,
        "csv_path": str(path),
    }


csv_paths = sorted(ARTIFACT_ROOT.rglob(CSV_PATTERN))
if not csv_paths:
    raise FileNotFoundError(f"No {CSV_PATTERN} files found under {ARTIFACT_ROOT.resolve()}")

frames = []
for csv_path in csv_paths:
    frame = pd.read_csv(csv_path)
    for key, value in metadata_from_csv_path(csv_path).items():
        frame[key] = value
    frames.append(frame)

sim_df = pd.concat(frames, ignore_index=True)

for column in SCORE_COLUMNS:
    if column in sim_df.columns:
        sim_df[column] = pd.to_numeric(sim_df[column], errors="coerce")

if "technical_valid" in sim_df.columns:
    sim_df["technical_valid_bool"] = sim_df["technical_valid"].astype(str).str.lower().eq("true")

sim_df["condition"] = sim_df["survey_block"].str.upper() + " / policy " + sim_df["policy"] + " / " + sim_df["pov"]

print(f"Loaded {len(sim_df)} rows from {len(csv_paths)} CSV files")
display(sim_df.head())

## Explore The Logged Data

First we check whether the simulation produced the expected blocks, policies, POVs, and episode slots. This is a data quality step, not a statistical test. If a block or episode is missing, all later comparisons have to be interpreted as incomplete.

In [ ]:
overview_columns = [
    "survey_block",
    "policy",
    "pov",
    "episode_slot",
    "episode_type",
    "run_label",
    "run_id",
    "episode_id",
    "terminal_status",
    "technical_valid",
]

available_overview_columns = [column for column in overview_columns if column in sim_df.columns]
display(sim_df[available_overview_columns].sort_values(["survey_block", "episode_slot", "run_id"]))

In [ ]:
status_columns = [column for column in ["technical_valid_bool", "terminal_status"] if column in sim_df.columns]

if status_columns:
    status_summary = (
        sim_df.groupby(["survey_block", "policy", "pov", "episode_slot"], dropna=False)[status_columns]
        .agg(lambda series: series.value_counts(dropna=False).to_dict())
        .reset_index()
    )
    display(status_summary)
else:
    print("No status columns found in the loaded metrics.")

Missing values are expected for some feature columns because not every episode contains the conditions needed to compute every metric. The high-level axes should usually be present for all valid episode runs.

In [ ]:
existing_score_columns = [column for column in SCORE_COLUMNS if column in sim_df.columns]

missingness = (
    sim_df[existing_score_columns]
    .isna()
    .mean()
    .rename("missing_fraction")
    .to_frame()
    .assign(non_missing_count=sim_df[existing_score_columns].notna().sum())
    .sort_values("missing_fraction", ascending=False)
)

display(missingness)

Now we summarize each condition. If there are multiple attempts for the same condition and episode slot, the table below keeps them as repeated logged runs and reports their mean, standard deviation, and count.

In [ ]:
axis_summary = (
    sim_df.groupby(["survey_block", "policy", "pov"], dropna=False)[AXIS_COLUMNS]
    .agg(["mean", "std", "count"])
    .sort_index()
)

display(axis_summary)

In [ ]:
episode_axis_summary = (
    sim_df.groupby(["survey_block", "policy", "pov", "episode_slot", "episode_type"], dropna=False)[AXIS_COLUMNS]
    .mean()
    .reset_index()
    .sort_values(["survey_block", "episode_slot"])
)

display(episode_axis_summary)

## Graphic Visualization

These plots are intentionally simple. We want fast checks that reveal whether one policy is consistently higher, whether a POV changes the score, and whether a single episode type dominates the result.

In [ ]:
condition_axis_means = (
    sim_df.groupby(["survey_block", "policy", "pov"], dropna=False)[AXIS_COLUMNS]
    .mean()
    .reset_index()
)
condition_axis_means["condition"] = (
    condition_axis_means["survey_block"].str.upper()
    + "\npolicy "
    + condition_axis_means["policy"]
    + "\n"
    + condition_axis_means["pov"]
)

plot_table = condition_axis_means.set_index("condition")[AXIS_COLUMNS]
ax = plot_table.plot(kind="bar", figsize=(12, 5), ylim=(0, 1), rot=0)
ax.set_title("Mean high-level simulation axes by condition")
ax.set_ylabel("Normalized score")
ax.set_xlabel("")
ax.legend(title="Axis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
episode_plot_table = episode_axis_summary.copy()
episode_plot_table["episode_condition"] = (
    episode_plot_table["survey_block"].str.upper()
    + " / "
    + episode_plot_table["episode_slot"].fillna("unknown")
)

ax = episode_plot_table.set_index("episode_condition")[AXIS_COLUMNS].plot(
    kind="bar",
    figsize=(14, 5),
    ylim=(0, 1),
    rot=45,
)
ax.set_title("Mean high-level axes by condition and episode")
ax.set_ylabel("Normalized score")
ax.set_xlabel("")
ax.legend(title="Axis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
feature_columns_present = [column for column in FEATURE_COLUMNS if column in sim_df.columns]

feature_heatmap = (
    sim_df.groupby(["survey_block", "episode_slot"], dropna=False)[feature_columns_present]
    .mean()
    .sort_index()
)

masked_values = np.ma.masked_invalid(feature_heatmap.to_numpy(dtype=float))
cmap = plt.cm.viridis.copy()
cmap.set_bad("lightgray")

fig, ax = plt.subplots(figsize=(14, max(4, len(feature_heatmap) * 0.45)))
image = ax.imshow(masked_values, aspect="auto", vmin=0, vmax=1, cmap=cmap)
ax.set_title("Feature means by survey block and episode")
ax.set_xticks(range(len(feature_heatmap.columns)))
ax.set_xticklabels(feature_heatmap.columns, rotation=45, ha="right")
ax.set_yticks(range(len(feature_heatmap.index)))
ax.set_yticklabels([" / ".join(map(str, index)) for index in feature_heatmap.index])
fig.colorbar(image, ax=ax, label="Normalized score")
plt.tight_layout()
plt.show()

display(feature_heatmap)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sim_df[AXIS_COLUMNS].plot(kind="box", ax=ax)
ax.set_title("Distribution of high-level simulation axes across logged runs")
ax.set_ylabel("Normalized score")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Statistical Analysis

The simulation data is not the same kind of sample as the survey data. A survey row is a person rating videos. A simulation row is a logged execution of a policy in a scripted episode. That means the simulation can tell us about objective benchmark behavior, but it cannot by itself estimate population-level human perception.

For this notebook we keep the analysis deliberately simple:

1. **Aggregate repeated runs** within the same survey block and episode slot. If there are multiple attempts, their mean is treated as the episode-level value for that policy/POV.
2. **Compare policies within the same POV**: Q1 vs Q2 for first person, and Q3 vs Q4 for third person.
3. **Compare POVs within the same policy**: Q1 vs Q3 for policy A, and Q2 vs Q4 for policy B.
4. **Report paired differences by episode slot**. The pairing matters because ep1, ep2, and ep3 represent different situations.
5. **Avoid strong significance claims when there are only three episode slots**. With ep1, ep2, and ep3, the useful result is the size and direction of the differences, not a precise p-value.

In short: this section checks whether the objective benchmark ranks policies and POVs consistently across the three scenario types.

In [ ]:
episode_condition = (
    sim_df.groupby(["survey_block", "policy", "pov", "episode_slot", "episode_type"], dropna=False)[AXIS_COLUMNS]
    .mean()
    .reset_index()
)
episode_condition["global_axis_mean"] = episode_condition[AXIS_COLUMNS].mean(axis=1)

METRICS_TO_COMPARE = ["global_axis_mean"] + AXIS_COLUMNS


def paired_block_comparison(left_block: str, right_block: str, label: str) -> pd.DataFrame:
    left = episode_condition[episode_condition["survey_block"] == left_block].set_index("episode_slot")
    right = episode_condition[episode_condition["survey_block"] == right_block].set_index("episode_slot")
    common_episode_slots = sorted(set(left.index).intersection(set(right.index)))

    rows = []
    for metric in METRICS_TO_COMPARE:
        differences = (left.loc[common_episode_slots, metric] - right.loc[common_episode_slots, metric]).dropna()
        rows.append(
            {
                "comparison": label,
                "left_block": left_block.upper(),
                "right_block": right_block.upper(),
                "metric": metric,
                "n_episode_pairs": int(differences.shape[0]),
                "mean_difference_left_minus_right": differences.mean(),
                "median_difference_left_minus_right": differences.median(),
                "left_wins": int((differences > 0).sum()),
                "ties": int((differences == 0).sum()),
                "right_wins": int((differences < 0).sum()),
                "episode_differences": differences.to_dict(),
            }
        )
    return pd.DataFrame(rows)


objective_comparisons = pd.concat(
    [
        paired_block_comparison("q1", "q2", "Policy A minus B, first-person POV"),
        paired_block_comparison("q3", "q4", "Policy A minus B, third-person POV"),
        paired_block_comparison("q1", "q3", "First-person minus third-person, policy A"),
        paired_block_comparison("q2", "q4", "First-person minus third-person, policy B"),
    ],
    ignore_index=True,
)

display(objective_comparisons)

In [ ]:
comparison_plot = objective_comparisons.pivot(
    index="comparison",
    columns="metric",
    values="mean_difference_left_minus_right",
)[METRICS_TO_COMPARE]

ax = comparison_plot.plot(kind="barh", figsize=(11, 6), xlim=(-1, 1))
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Mean paired differences across episode slots")
ax.set_xlabel("Mean difference")
ax.set_ylabel("")
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

The next table checks how the low-level features relate to the high-level axes. This is useful because if a benchmark axis is driven by only one feature, or if two axes are almost identical, we should know that before using the benchmark as a predictor of survey outcomes.

In [ ]:
correlation_columns = [column for column in SCORE_COLUMNS if column in sim_df.columns and sim_df[column].notna().sum() > 1]
correlation_matrix = sim_df[correlation_columns].corr()

axis_feature_correlations = correlation_matrix.loc[
    [column for column in AXIS_COLUMNS if column in correlation_matrix.index],
    [column for column in FEATURE_COLUMNS if column in correlation_matrix.columns],
]

display(axis_feature_correlations)

fig, ax = plt.subplots(figsize=(14, 4))
masked_values = np.ma.masked_invalid(axis_feature_correlations.to_numpy(dtype=float))
cmap = plt.cm.coolwarm.copy()
cmap.set_bad("lightgray")
image = ax.imshow(masked_values, aspect="auto", vmin=-1, vmax=1, cmap=cmap)
ax.set_title("Correlation between high-level axes and low-level features")
ax.set_xticks(range(len(axis_feature_correlations.columns)))
ax.set_xticklabels(axis_feature_correlations.columns, rotation=45, ha="right")
ax.set_yticks(range(len(axis_feature_correlations.index)))
ax.set_yticklabels(axis_feature_correlations.index)
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## Interpretation Notes

Use this notebook to answer simulation-side questions such as:

- Did every Q block and episode type run successfully?
- Which policy has higher objective scores within the same POV?
- Does the camera/POV change the objective scores for the same policy?
- Are the benchmark axes supported by multiple features or dominated by a small number of features?

Do not use this notebook alone to claim that people will prefer one policy. That claim belongs in the survey notebook. The bridge between the two is the next modeling step: compare the objective simulation distributions with the human response distributions and check whether the benchmark ranking predicts the survey ranking.